# Лабораторная работа №6
## Высокопроизводительная обработка данных в NumPy
**Вариант 16: Лазерные измерения**

**Цель работы:** освоение методов высокопроизводительной обработки данных
в Python с использованием библиотеки NumPy: векторизованные вычисления,
структурированные массивы, группировка, оконные функции, лаговые признаки,
IQR-фильтрация, частотный анализ.

**Описание варианта.** Контроль параметров промышленных лазеров: мощность,
длина волны, частота импульсов, статусы ошибок оптики.

**Поля датасета:**
- `ts` (int32): время замера
- `beam_id` (int16): ID луча
- `power` (float32): мощность излучения, Вт
- `wave` (float32): длина волны, нм
- `pulse` (float32): частота импульсов, Гц
- `err` (uint8): код ошибки оптики (0–15)

**Акцент по варианту:** маска `err == 0` применяется перед всеми
аналитическими расчётами (записи с ненулевым кодом ошибки считаются
недостоверными).


In [ ]:
import os
import numpy as np

print("NumPy version:", np.__version__)


## Этап 1. Загрузка и подготовка типов

Читаем CSV в структурированный массив `numpy.genfromtxt`. Считаем число
строк, объём занятой памяти, количество NaN/Inf в числовых полях.
При доле «битых» значений выше 3% выдаём предупреждение.


In [ ]:
DATA_PATH = "data_variant_16.csv"

# Универсальная загрузка: локально или из Colab-загрузчика
if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files  # type: ignore
        print("Файл не найден локально — выберите его в окне загрузки Colab.")
        uploaded = files.upload()
        DATA_PATH = list(uploaded.keys())[0]
    except Exception as exc:
        raise FileNotFoundError(
            "Положите data_variant_16.csv рядом с ноутбуком"
        ) from exc

# Структурированный dtype под спецификацию варианта
dtype = np.dtype([
    ("ts",      np.int32),
    ("beam_id", np.int16),
    ("power",   np.float32),
    ("wave",    np.float32),
    ("pulse",   np.float32),
    ("err",     np.uint8),
])

print("Чтение CSV ...")
data = np.genfromtxt(
    DATA_PATH,
    delimiter=",",
    dtype=dtype,
    skip_header=1,
    invalid_raise=False,   # пропускаем битые строки (неполные/лишние колонки)
)
# genfromtxt может вернуть 0-d массив если строк 1 — нормализуем
data = np.atleast_1d(data)
n_rows = data.shape[0]
nbytes = data.nbytes
mb = nbytes / (1024 * 1024)
print(f"Прочитано строк:    {n_rows}")
print(f"Объём в памяти:     {nbytes} байт ({mb:.2f} МБ)")
print(f"dtype массива:      {data.dtype}")


In [ ]:
# Подсчёт NaN/Inf в числовых полях
numeric_fields = ["power", "wave", "pulse"]
total_bad = 0
total_cells = 0
for f in numeric_fields:
    arr = data[f]
    nans = int(np.isnan(arr).sum())
    infs = int(np.isinf(arr).sum())
    total_bad += nans + infs
    total_cells += arr.size
    print(f"  {f}: NaN={nans}, Inf={infs}")

share_bad = total_bad / total_cells if total_cells else 0.0
print(f"Доля NaN/Inf по числовым полям: {share_bad:.4%}")
if share_bad > 0.03:
    print("ВНИМАНИЕ: доля NaN/Inf превышает 3%, данные требуют очистки!")
else:
    print("Доля NaN/Inf в норме (<= 3%).")

# Сохраняем исходный массив
np.save("stage1_raw.npy", data)
print("Сохранено: stage1_raw.npy")


## Этап 2. Векторизованная фильтрация и очистка

По заданию для варианта 16:
- `power < 0` — нефизичная мощность, заменяем на 0;
- `pulse > max_val` — частоту ограничиваем «паспортным максимумом»;
- `wave` — обрезаем по `[Q5, Q95]`.

Дополнительно перед основной очисткой обрабатываем NaN/Inf в числовых
полях: заменяем их на медиану соответствующего столбца (np.nanmedian).
Это нейтральная замена, не искажающая распределение.

`max_val` для частоты импульсов берём как 99-й процентиль положительных
значений `pulse` (инженерный «паспортный максимум»).

Используем только булевы маски, `np.where`, `np.clip` — без циклов.


In [ ]:
# 2.0 Замена NaN/Inf на медиану столбца (защита от падения дальнейших шагов)
for f in numeric_fields:
    arr = data[f].astype(np.float64)
    bad = np.isnan(arr) | np.isinf(arr)
    if bad.any():
        med = float(np.nanmedian(np.where(np.isinf(arr), np.nan, arr)))
        data[f] = np.where(bad, med, arr).astype(data[f].dtype)
        print(f"  {f}: заменено NaN/Inf на медиану ({med:.4f}): {int(bad.sum())}")

# 2.1 Аномалии — считаем количество и долю до очистки
mask_power_neg  = data["power"] < 0
mask_wave_neg   = data["wave"] < 0
n_power_neg = int(mask_power_neg.sum())
n_wave_neg  = int(mask_wave_neg.sum())

positive_pulse = data["pulse"][data["pulse"] > 0]
max_val_pulse = float(np.nanquantile(positive_pulse, 0.99))
print(f"Паспортный max_val для pulse (q99): {max_val_pulse:.4f} Гц")
mask_pulse_high = data["pulse"] > max_val_pulse
n_pulse_high = int(mask_pulse_high.sum())

q5_wave  = float(np.nanquantile(data["wave"], 0.05))
q95_wave = float(np.nanquantile(data["wave"], 0.95))
mask_wave_out = (data["wave"] < q5_wave) | (data["wave"] > q95_wave)
n_wave_out = int(mask_wave_out.sum())

mask_anomaly = mask_power_neg | mask_pulse_high | mask_wave_out
n_anomaly = int(mask_anomaly.sum())

print(f"power<0:           {n_power_neg:>10d}  ({n_power_neg/n_rows:.4%})")
print(f"pulse>max_val:     {n_pulse_high:>10d}  ({n_pulse_high/n_rows:.4%})")
print(f"wave не в [Q5,Q95]:{n_wave_out:>10d}  ({n_wave_out/n_rows:.4%})")
print(f"wave<0 (доп.):     {n_wave_neg:>10d}  ({n_wave_neg/n_rows:.4%})")
print(f"Всего аномальных:  {n_anomaly:>10d}  ({n_anomaly/n_rows:.4%})")

# 2.2 Очистка строго по заданию
data["power"] = np.where(data["power"] < 0, 0.0, data["power"])
data["pulse"] = np.where(
    data["pulse"] > max_val_pulse,
    np.float32(max_val_pulse),
    data["pulse"],
)
data["wave"] = np.clip(data["wave"], q5_wave, q95_wave)

# Контроль после очистки
print("Контроль после очистки:")
print(f"  min power: {data['power'].min():.4f}")
print(f"  max pulse: {data['pulse'].max():.4f}")
print(f"  диапазон wave: [{data['wave'].min():.4f}; {data['wave'].max():.4f}]")


## Этап 2.5. Маска `err == 0` (акцент варианта)

Записи с ненулевым кодом ошибки оптики считаются недостоверными — их
исключаем перед групповыми и оконными расчётами.


In [ ]:
mask_ok = data["err"] == 0
data_ok = data[mask_ok].copy()
print(f"Строк с err == 0: {data_ok.shape[0]} ({data_ok.shape[0]/n_rows:.4%})")

# Сортировка по времени для корректной работы окна и лагов
order = np.argsort(data_ok["ts"], kind="stable")
data_ok = data_ok[order]


## Этап 3a. Группировка по `beam_id` + Z-score нормализация `power`

Группируем через `np.unique(beam_id)` + цикл **по группам** (не по
строкам). Считаем мощность каждой группы (count), среднюю мощность,
СКО и максимальную частоту импульсов. Затем — Z-score нормализация
`power` внутри каждой группы.


In [ ]:
beam_ids, inverse, counts = np.unique(
    data_ok["beam_id"], return_inverse=True, return_counts=True,
)
n_groups = beam_ids.shape[0]
print(f"Количество групп (beam_id): {n_groups}")

means_power = np.zeros(n_groups, dtype=np.float64)
stds_power  = np.zeros(n_groups, dtype=np.float64)
max_pulses  = np.zeros(n_groups, dtype=np.float64)

for i, g in enumerate(beam_ids):
    sel = data_ok["beam_id"] == g
    p = data_ok["power"][sel].astype(np.float64)
    means_power[i] = p.mean()
    stds_power[i]  = p.std()
    max_pulses[i]  = data_ok["pulse"][sel].astype(np.float64).max()

print("Топ-5 групп по средней мощности:")
top5 = np.argsort(-means_power)[:5]
for i in top5:
    print(
        f"  beam_id={beam_ids[i]:>3d}: count={counts[i]:>7d}, "
        f"mean_power={means_power[i]:8.4f}, std_power={stds_power[i]:8.4f}, "
        f"max_pulse={max_pulses[i]:8.4f}"
    )

# Z-score через broadcasting (без поэлементных циклов)
group_mean = means_power[inverse]
group_std  = stds_power[inverse]
safe_std = np.where(group_std > 1e-8, group_std, 1.0)
power_z = (data_ok["power"].astype(np.float64) - group_mean) / safe_std
print(
    f"Z-score power: mean={power_z.mean():.6f} (≈0), "
    f"std={power_z.std():.6f} (≈1)"
)

# Сохраним нормализованный массив (отдельным файлом)
np.save("stage3_filtered.npy", data_ok)
np.save("stage3_power_zscore.npy", power_z.astype(np.float32))
print("Сохранено: stage3_filtered.npy, stage3_power_zscore.npy")


## Этап 3b (по заданию). Скользящее окно `k=25` и `np.diff(pulse)`

Скользящее среднее по `power` считаем через формулу кумулятивной суммы
(O(N), без циклов). Первые `k-1` значений заполняем `NaN` через `np.pad`
/ конкатенацию (требование: «обрежьте или дополните»).

Дополнительно считаем `np.diff(pulse)` — скорость изменения частоты
между соседними замерами — и добавляем как новое поле `pulse_diff`
в структурированный массив.


In [ ]:
K = 25  # размер окна по варианту

x = data_ok["power"].astype(np.float64)
cumsum = np.concatenate(([0.0], np.cumsum(x)))
mavg = (cumsum[K:] - cumsum[:-K]) / K
power_mavg = np.concatenate((np.full(K - 1, np.nan), mavg))
print(f"Длина power_mavg: {power_mavg.shape[0]} (NaN в начале: {K - 1})")
print(f"Скользящее среднее: min={np.nanmin(power_mavg):.4f}, "
      f"max={np.nanmax(power_mavg):.4f}, mean={np.nanmean(power_mavg):.4f}")

# np.diff(pulse) — длина N-1, дополним 0 в начале для соответствия размеру
pulse_diff = np.diff(data_ok["pulse"].astype(np.float64))
pulse_diff_full = np.concatenate(([0.0], pulse_diff))
print(f"np.diff(pulse): min={pulse_diff.min():.4f}, "
      f"max={pulse_diff.max():.4f}, mean={pulse_diff.mean():.6f}")

# Расширяем структурированный массив: добавляем pulse_diff и power_mavg
new_dtype = np.dtype(data_ok.dtype.descr + [
    ("pulse_diff", np.float32),
    ("power_mavg", np.float32),
])
ext = np.zeros(data_ok.shape[0], dtype=new_dtype)
for f in data_ok.dtype.names:
    ext[f] = data_ok[f]
ext["pulse_diff"] = pulse_diff_full.astype(np.float32)
ext["power_mavg"] = power_mavg.astype(np.float32)
data_ok = ext
print(f"Поля массива после этапа 3b: {data_ok.dtype.names}")


## Этап 4. Производные признаки (Feature Engineering)

Вычисляем два новых показателя, логически вытекающих из предметной
области (промышленный лазер):

1. `energy_per_pulse = power / pulse` — приближённая энергия одного
   импульса (Вт делим на частоту импульсов в Гц = Дж/импульс).
2. `eff_emission = power / wave` — удельная мощность на единицу длины
   волны (косвенный показатель эффективности излучения).

Деления выполняем с защитой `1e-8` в знаменателе. Возникшие `inf`/`nan`
заменяем на нейтральное значение 0.


In [ ]:
power_arr = data_ok["power"].astype(np.float64)
wave_arr  = data_ok["wave"].astype(np.float64)
pulse_arr = data_ok["pulse"].astype(np.float64)

denom_pulse = np.where(pulse_arr > 0, pulse_arr, 1e-8)
denom_wave  = np.where(wave_arr  > 0, wave_arr,  1e-8)

energy_per_pulse = power_arr / denom_pulse
eff_emission     = power_arr / denom_wave


def sanitize(arr):
    """Заменяет inf/-inf/nan на 0 (нейтральное значение)."""
    arr = np.where(np.isinf(arr), 0.0, arr)
    arr = np.where(np.isnan(arr), 0.0, arr)
    return arr


energy_per_pulse = sanitize(energy_per_pulse)
eff_emission     = sanitize(eff_emission)

print(f"energy_per_pulse: mean={energy_per_pulse.mean():.4f}, "
      f"max={energy_per_pulse.max():.4f}, min={energy_per_pulse.min():.4f}")
print(f"eff_emission:     mean={eff_emission.mean():.4f}, "
      f"max={eff_emission.max():.4f}, min={eff_emission.min():.4f}")

# Добавляем оба поля в массив
new_dtype = np.dtype(data_ok.dtype.descr + [
    ("energy_per_pulse", np.float32),
    ("eff_emission",     np.float32),
])
ext = np.zeros(data_ok.shape[0], dtype=new_dtype)
for f in data_ok.dtype.names:
    ext[f] = data_ok[f]
ext["energy_per_pulse"] = energy_per_pulse.astype(np.float32)
ext["eff_emission"]     = eff_emission.astype(np.float32)
data_ok = ext
print(f"Поля массива после этапа 4: {data_ok.dtype.names}")


## Этап 5. Условная агрегация по группам

Для каждой группы (`beam_id`) считаем статистики `power` только по
подмножеству «активных» замеров — где `pulse` строго больше медианы
группы. Маска формируется составной без циклов по строкам.

Итог — массив формы `(N_groups, 3)`: `[group_id, mean, median]`.
Дополнительно выводим 90-й процентиль для каждой группы.


In [ ]:
beam_ids, inverse = np.unique(data_ok["beam_id"], return_inverse=True)
n_groups = beam_ids.shape[0]

# Медианы pulse по группам
group_pulse_median = np.zeros(n_groups)
for i, g in enumerate(beam_ids):
    sel = data_ok["beam_id"] == g
    group_pulse_median[i] = np.median(data_ok["pulse"][sel])

# Составная маска: «активный режим» (pulse выше медианы своей группы)
mask_active = data_ok["pulse"] > group_pulse_median[inverse]
print(f"Активных строк (pulse > медианы группы): "
      f"{mask_active.sum()} ({mask_active.mean():.4%})")

table = np.zeros((n_groups, 3), dtype=np.float64)
p90 = np.zeros(n_groups, dtype=np.float64)
for i, g in enumerate(beam_ids):
    sel = (data_ok["beam_id"] == g) & mask_active
    if not sel.any():
        table[i] = (g, 0.0, 0.0)
        p90[i] = 0.0
        continue
    p = data_ok["power"][sel]
    table[i, 0] = g
    table[i, 1] = p.mean()
    table[i, 2] = np.median(p)
    p90[i] = np.quantile(p, 0.9)

print("Таблица условной агрегации (формат [group_id, mean, median]):")
print(f"  Форма: {table.shape}")
print("  Первые 10 строк:")
for row in table[:10]:
    print(f"    beam_id={int(row[0]):>3d}: mean={row[1]:8.4f}, "
          f"median={row[2]:8.4f}")
print("90-й процентиль power для первых 10 групп:")
for g, q in zip(beam_ids[:10].tolist(), p90[:10].tolist()):
    print(f"    beam_id={g:>3d}: p90={q:8.4f}")

np.save("stage5_conditional_agg.npy", table)
print("Сохранено: stage5_conditional_agg.npy")


## Этап 6. Лаговые признаки и анализ временных сдвигов

Создаём признак «предыдущее значение» (lag=1) для `power` через
`np.roll`, считаем разницу `current - lag`, обрабатываем граничный
элемент (присваиваем 0). Считаем долю записей, где значение выросло /
упало / осталось неизменным, через `np.sign` + `np.unique`.


In [ ]:
power_cur = data_ok["power"].astype(np.float64)
power_lag1 = np.roll(power_cur, 1)
delta = power_cur - power_lag1
delta[0] = 0.0  # граничный элемент
delta = np.nan_to_num(delta, nan=0.0, posinf=0.0, neginf=0.0)

n = data_ok.shape[0]
ups   = int((delta > 0).sum())
downs = int((delta < 0).sum())
same  = int((delta == 0).sum())
print(f"Выросло:        {ups:>10d}  ({ups/n:.4%})")
print(f"Упало:          {downs:>10d}  ({downs/n:.4%})")
print(f"Без изменений:  {same:>10d}  ({same/n:.4%})")

signs = np.sign(delta).astype(np.int64)
vals, cnts = np.unique(signs, return_counts=True)
print("Распределение знаков (np.unique):")
for v, c in zip(vals.tolist(), cnts.tolist()):
    label = {-1: "падение", 0: "без изм.", 1: "рост"}.get(v, str(v))
    print(f"  sign={v:>2d} ({label:>8s}): {c} ({c/n:.4%})")

# Альтернативно np.bincount: индексы должны быть >= 0, поэтому смещаем
shifted = signs + 1  # -1,0,1 -> 0,1,2
bc = np.bincount(shifted, minlength=3)
print(f"np.bincount по signs+1: падение={bc[0]}, без изм.={bc[1]}, рост={bc[2]}")

# Добавим лаговый признак как новое поле
new_dtype = np.dtype(data_ok.dtype.descr + [
    ("power_delta", np.float32),
])
ext = np.zeros(data_ok.shape[0], dtype=new_dtype)
for f in data_ok.dtype.names:
    ext[f] = data_ok[f]
ext["power_delta"] = delta.astype(np.float32)
data_ok = ext


## Этап 7. Групповая робастная замена выбросов (IQR)

Внутри каждой группы вычисляем `Q1`, `Q3`, `IQR = Q3 - Q1`, границы
`[Q1 - 1.5·IQR, Q3 + 1.5·IQR]`. Значения вне границ считаются
выбросами и заменяются на медиану той же группы. Цикл — по группам,
не по строкам исходного массива.


In [ ]:
p = data_ok["power"].astype(np.float64).copy()
beam_ids, inverse = np.unique(data_ok["beam_id"], return_inverse=True)
n_groups = beam_ids.shape[0]

replaced_per_group = np.zeros(n_groups, dtype=np.int64)
total_replaced = 0
for i, g in enumerate(beam_ids):
    sel = data_ok["beam_id"] == g
    p_g = p[sel]
    if p_g.size < 4:
        continue
    q1 = np.quantile(p_g, 0.25)
    q3 = np.quantile(p_g, 0.75)
    iqr = q3 - q1
    low  = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr
    med  = float(np.median(p_g))
    out_mask = (p_g < low) | (p_g > high)
    n_out = int(out_mask.sum())
    if n_out:
        p[sel] = np.where(out_mask, med, p_g)
        replaced_per_group[i] = n_out
        total_replaced += n_out

print(f"Топ-10 групп по числу замен (IQR):")
top = np.argsort(-replaced_per_group)[:10]
for i in top:
    if replaced_per_group[i] == 0:
        continue
    print(f"  beam_id={beam_ids[i]:>3d}: заменено {replaced_per_group[i]}")
print(f"Всего заменено значений power: {total_replaced}")
print(f"Доля изменённых записей: {total_replaced/data_ok.shape[0]:.4%}")

data_ok["power"] = p.astype(np.float32)


## Этап 8. Проверка согласованности и логической целостности

Правила предметной области (лазер):
- `power >= 0` — мощность не отрицательна;
- `wave > 0`  — длина волны строго положительна;
- `pulse >= 0` — частота импульсов не отрицательна;
- `err in [0..15]` — код ошибки оптики в допустимом диапазоне.

Составная маска формируется без циклов. Некорректные коды ошибок
(`err > 15`) заменяются на эталон `0` через `np.where`.


In [ ]:
mask_invalid = (
    (data_ok["power"] < 0)
    | (data_ok["wave"] <= 0)
    | (data_ok["pulse"] < 0)
    | (data_ok["err"] > 15)
)
n_invalid = int(mask_invalid.sum())
print(f"Записей с нарушениями целостности: "
      f"{n_invalid} ({n_invalid/data_ok.shape[0]:.4%})")

# Замена некорректных err > 15 на 0 (эталон)
err_before = data_ok["err"].copy()
data_ok["err"] = np.where(data_ok["err"] > 15, 0, data_ok["err"]).astype(np.uint8)
n_err_fixed = int((err_before > 15).sum())
print(f"Заменено err > 15 -> 0: {n_err_fixed}")


## Этап 9. Частотный анализ и сжатие редких категорий

Анализируем распределение поля `err` (исходные данные до этапа 8 —
чтобы увидеть редкие «битые» коды). Категории с долей менее 1%
объединяем в одну — `OTHER` с кодом `255`.


In [ ]:
err_orig = data["err"].copy()  # исходные коды до замены на этапе 8
n_total = err_orig.size
vals_e, counts_e = np.unique(err_orig, return_counts=True)
freqs = counts_e / n_total
print(f"Уникальных кодов err в исходных данных: {vals_e.size}")

print("Частоты топ-10 категорий:")
order_top = np.argsort(-counts_e)[:10]
for idx in order_top:
    print(f"  err={int(vals_e[idx]):>3d}: count={counts_e[idx]:>8d} "
          f"({freqs[idx]:.4%})")

rare_codes = vals_e[freqs < 0.01]
print(f"Редких категорий (доля < 1%): {rare_codes.size}")

err_compressed = np.where(
    np.isin(err_orig, rare_codes), 255, err_orig
).astype(np.uint8)
n_replaced_rare = int(np.isin(err_orig, rare_codes).sum())
print(f"Заменено в OTHER (255): {n_replaced_rare} ({n_replaced_rare/n_total:.4%})")

vals_after = np.unique(err_compressed)
print(f"Категорий после сжатия: {vals_after.size}")
print(f"Список оставшихся категорий: {vals_after.tolist()}")


## Финальный итог

- Исходный массив сохранён в `stage1_raw.npy`.
- Очищенный массив (с фильтром `err == 0`) — в `stage3_filtered.npy`.
- Z-score нормализованная мощность — в `stage3_power_zscore.npy`.
- Таблица условной агрегации — в `stage5_conditional_agg.npy`.

Все этапы выполнены с использованием только векторизованных операций
NumPy и булевых масок; циклы применялись исключительно по группам,
но не по строкам исходного массива.


In [ ]:
print("=" * 60)
print("Итоговое состояние массива data_ok:")
print(f"  строк:  {data_ok.shape[0]}")
print(f"  полей:  {len(data_ok.dtype.names)}")
print(f"  поля:   {data_ok.dtype.names}")
print(f"  память: {data_ok.nbytes/1024/1024:.2f} МБ")
print("Лабораторная работа выполнена.")
